In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, get_scorer_names


In [2]:
df = pd.read_csv('../data/processed/model_training_data.csv')

target = 'race_is_winner'

numeric_features = ['year','fp1_lap_count', 'fp1_lap_mean', 'fp1_lap_std', 'fp1_lap_best', 'fp1_lap_median', 'fp1_air_temp_mean', 'fp1_air_temp_std', 
                    'fp1_track_temp_mean', 'fp1_track_temp_std', 'fp1_humidity_mean', 'fp1_humidity_std', 'fp1_wind_speed_mean', 'fp1_wind_speed_std',
                    'fp1_rain_any', 'fp1_rain_samples_ratio', 'fp1_yellow_count', 'fp1_red_count', 'fp1_vsc_deployed_count', 'fp1_sc_deployed_count',
                    'fp1_disruption_ratio', 'fp1_best_free_practice_sec', 'fp1_free_practice_delta_to_best_lap_sec', 'fp1_free_practice_percent_of_best_lap_sec', 
                    'fp2_lap_count', 'fp2_lap_mean', 'fp2_lap_std', 'fp2_lap_best', 'fp2_lap_median', 'fp2_air_temp_mean', 'fp2_air_temp_std', 'fp2_track_temp_mean', 
                    'fp2_track_temp_std', 'fp2_humidity_mean', 'fp2_humidity_std', 'fp2_wind_speed_mean', 'fp2_wind_speed_std', 'fp2_rain_any', 'fp2_rain_samples_ratio', 
                    'fp2_yellow_count', 'fp2_red_count', 'fp2_vsc_deployed_count', 'fp2_sc_deployed_count', 'fp2_disruption_ratio', 'fp2_best_free_practice_sec', 
                    'fp2_free_practice_delta_to_best_lap_sec', 'fp2_free_practice_percent_of_best_lap_sec', 'fp3_lap_count', 'fp3_lap_mean', 'fp3_lap_std',
                    'fp3_lap_best', 'fp3_lap_median', 'fp3_air_temp_mean', 'fp3_air_temp_std', 'fp3_track_temp_mean', 'fp3_track_temp_std', 'fp3_humidity_mean', 
                    'fp3_humidity_std', 'fp3_wind_speed_mean', 'fp3_wind_speed_std', 'fp3_rain_any', 'fp3_rain_samples_ratio', 'fp3_yellow_count', 
                    'fp3_red_count', 'fp3_vsc_deployed_count', 'fp3_sc_deployed_count', 'fp3_disruption_ratio', 'fp3_best_free_practice_sec', 
                    'fp3_free_practice_delta_to_best_lap_sec', 'fp3_free_practice_percent_of_best_lap_sec', 'quali_lap_count', 'quali_lap_mean', 
                    'quali_lap_std', 'quali_lap_best', 'quali_lap_median', 'quali_air_temp_mean', 'quali_air_temp_std', 'quali_track_temp_mean', 
                    'quali_track_temp_std', 'quali_humidity_mean', 'quali_humidity_std', 'quali_wind_speed_mean', 'quali_wind_speed_std', 'quali_rain_any',
                    'quali_rain_samples_ratio', 'quali_yellow_count', 'quali_red_count', 'quali_vsc_deployed_count', 'quali_sc_deployed_count', 'quali_disruption_ratio', 
                    'quali_reached_q1', 'quali_reached_q2', 'quali_reached_q3', 'quali_best_quali_seconds', 'quali_quali_delta_to_pole', 'quali_quali_percent_of_pole', 'quali_quali_finish_position']

categorical_features = ['grand_prix', 'driver_id', 'abbreviation', 'team_name', 'team_id']

features = numeric_features + categorical_features 

X = df[features]
y = df[target]

In [3]:
# preprocessing

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocess = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop')

# train/test split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=.8, test_size=.2, random_state=1, )

In [4]:
def quick_model_evaluation(df, model, target="race_is_winner", group_col="event_id", n_splits=3):
    # Make a copy so we do not change the original dataframe
    data = df.copy()

    # Keep only rows where the target exists
    data = data.dropna(subset=[target])

    # Remove columns that should not be used as features
    exclude_cols = [target, group_col, "row_id"]
    X = data.drop(columns=[col for col in exclude_cols if col in data.columns])

    # Keep only numeric columns for a quick test
    X = X.select_dtypes(include=["number"]).fillna(0)

    y = data[target].astype(int)
    groups = data[group_col]

    gkf = GroupKFold(n_splits=n_splits)

    winner_pick_scores = []
    row_accuracy_scores = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        test_groups = groups.iloc[test_idx]

        model.fit(X_train, y_train)

        # Probability that each driver is the winner
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = (y_proba >= 0.5).astype(int)

        # Regular row-level accuracy
        row_acc = accuracy_score(y_test, y_pred)
        row_accuracy_scores.append(row_acc)

        # Event-level winner pick:
        # choose the driver with the highest predicted probability in each event
        fold_eval = pd.DataFrame({
            "event_id": test_groups.values,
            "y_true": y_test.values,
            "y_proba": y_proba
        })

        predicted_winners = (
            fold_eval.sort_values(["event_id", "y_proba"], ascending=[True, False])
                     .groupby("event_id", as_index=False)
                     .first()
        )

        winner_pick_acc = predicted_winners["y_true"].mean()
        winner_pick_scores.append(winner_pick_acc)

        print(f"Fold {fold}: row_accuracy={row_acc:.4f}, winner_pick_accuracy={winner_pick_acc:.4f}")

    print("\nAverage row accuracy:", sum(row_accuracy_scores) / len(row_accuracy_scores))
    print("Average winner pick accuracy:", sum(winner_pick_scores) / len(winner_pick_scores))

    return {
        "mean_row_accuracy": sum(row_accuracy_scores) / len(row_accuracy_scores),
        "mean_winner_pick_accuracy": sum(winner_pick_scores) / len(winner_pick_scores)
    }

In [5]:
df['race_is_winner'].unique()

array([0, 1])

In [6]:
# evaluate potential models

quick_model_evaluation(df, RandomForestClassifier(), target="race_is_winner")
quick_model_evaluation(df, GradientBoostingClassifier(), target="race_is_winner")

Fold 1: row_accuracy=0.9571, winner_pick_accuracy=1.0000
Fold 2: row_accuracy=0.9857, winner_pick_accuracy=1.0000
Fold 3: row_accuracy=0.9786, winner_pick_accuracy=1.0000

Average row accuracy: 0.9738095238095239
Average winner pick accuracy: 1.0
Fold 1: row_accuracy=1.0000, winner_pick_accuracy=1.0000
Fold 2: row_accuracy=1.0000, winner_pick_accuracy=1.0000
Fold 3: row_accuracy=1.0000, winner_pick_accuracy=1.0000

Average row accuracy: 1.0
Average winner pick accuracy: 1.0


{'mean_row_accuracy': 1.0, 'mean_winner_pick_accuracy': np.float64(1.0)}